# Notebook 02: Dynamic Routing with Conditional Edges

Welcome back! In Notebook 01, you learned to build linear workflows. Now, you'll learn how to make your graphs **intelligent** by adding decision-making capabilities.

## Learning Objectives

By the end of this notebook, you will be able to:

1. Understand conditional edges and routing logic
2. Create routing functions that make decisions based on state
3. Build graphs with multiple execution paths
4. Handle branching workflows
5. Debug and visualize complex graph flows

## Prerequisites

- Completed Notebook 01
- Understanding of StateGraph, nodes, and edges
- Basic Python conditional logic (if/elif/else)

## Estimated Time

90-120 minutes

## Complexity Level

Beginner-Intermediate (2/5)

## 1. Introduction: Why Conditional Routing?

### The Challenge

In Notebook 01, we built **linear** workflows where execution always followed the same path:

```
node_a → node_b → node_c → END
```

But real-world workflows often need to **branch** based on intermediate results:

- Customer support: Route to different teams based on issue type
- Content moderation: Different actions for safe vs. unsafe content
- Data validation: Different handling for valid vs. invalid data
- Error handling: Retry vs. fallback vs. exit based on error type

### The Solution: Conditional Edges

**Conditional edges** allow graphs to make decisions at runtime based on the current state.

Instead of a fixed path, you define a **routing function** that:
1. Examines the current state
2. Returns the name of the next node to execute
3. Or returns `END` to terminate

### Flow Comparison

**Linear (Notebook 01):**
```
START → process → format → END
```

**Conditional (Notebook 02):**
```
START → classify → decision?
                     ├─ if spam → delete → END
                     ├─ if urgent → escalate → END
                     └─ if normal → process → END
```

## 2. Setup

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from typing import Literal

print("✅ Imports successful!")
print("🔀 Ready to build conditional workflows!")

d:\AI\LangGraph-Project\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


✅ Imports successful!
🔀 Ready to build conditional workflows!


> **Note**: `set_entry_point()` and `set_finish_point()` are deprecated. Use `add_edge(START, "node")` and `add_edge("node", END)` instead — which is what all examples in this series use.

## 3. Example 1: Content Classifier

Let's build a simple content classifier that routes text to different processing paths based on a score.

### Flow Diagram

```
START → analyze_content → route_content?
                              ├─ if score >= 0.7 → approve → END
                              ├─ if score < 0.3 → reject → END
                              └─ else → review → END
```

In [2]:
# Define state
class ContentState(TypedDict):
    text: str
    score: float  # 0.0 to 1.0
    category: str
    message: str

# Node: Analyze content and assign a score
def analyze_content(state: ContentState) -> dict:
    """Simulate content analysis with a simple scoring system."""
    print(f"🔍 Analyzing content: '{state['text'][:50]}...'")
    
    # Simple scoring: count "good" vs "bad" keywords
    text_lower = state["text"].lower()
    good_words = ["great", "excellent", "amazing", "helpful", "good"]
    bad_words = ["spam", "scam", "fake", "terrible", "hate"]
    
    good_count = sum(word in text_lower for word in good_words)
    bad_count = sum(word in text_lower for word in bad_words)
    
    # Calculate score
    if good_count + bad_count == 0:
        score = 0.5  # Neutral
    else:
        score = good_count / (good_count + bad_count)
    
    print(f"📊 Content score: {score:.2f}")
    return {"score": score}

# Routing function: Decides which path to take
def route_content(state: ContentState) -> Literal["approve", "reject", "review"]:
    """Route content based on score."""
    score = state["score"]
    
    if score >= 0.7:
        print("✅ Routing to APPROVE")
        return "approve"
    elif score < 0.3:
        print("❌ Routing to REJECT")
        return "reject"
    else:
        print("⚠️  Routing to REVIEW")
        return "review"

# Terminal nodes for each path
def approve_content(state: ContentState) -> dict:
    """Approve the content."""
    print("✅ Content APPROVED")
    return {
        "category": "approved",
        "message": "Content approved for publication"
    }

def reject_content(state: ContentState) -> dict:
    """Reject the content."""
    print("❌ Content REJECTED")
    return {
        "category": "rejected",
        "message": "Content does not meet quality standards"
    }

def review_content(state: ContentState) -> dict:
    """Flag for manual review."""
    print("⚠️  Content flagged for REVIEW")
    return {
        "category": "review",
        "message": "Content requires manual review"
    }

### Build the Graph with Conditional Edges

Notice the new method: `add_conditional_edges()`

In [3]:
# Build graph
content_graph = StateGraph(ContentState)

# Add nodes
content_graph.add_node("analyze", analyze_content)
content_graph.add_node("approve", approve_content)
content_graph.add_node("reject", reject_content)
content_graph.add_node("review", review_content)

# Add edges
content_graph.add_edge(START, "analyze")

# CONDITIONAL EDGE: The key new concept!
content_graph.add_conditional_edges(
    "analyze",           # Source node
    route_content,       # Routing function
    # The routing function returns one of these node names:
    # - "approve", "reject", or "review"
)

# Each terminal node goes to END
content_graph.add_edge("approve", END)
content_graph.add_edge("reject", END)
content_graph.add_edge("review", END)

# Compile
content_app = content_graph.compile()

print("✅ Content classifier ready!")

✅ Content classifier ready!


In [ ]:
import base64
from IPython.display import Image, display


mermaid_text = content_app.get_graph().draw_mermaid()

# Encode the mermaid syntax as base64 and use mermaid.ink as an image source
graph_bytes  = mermaid_text.encode("utf-8")
base64_str   = base64.urlsafe_b64encode(graph_bytes).decode("utf-8")
url          = f"https://mermaid.ink/img/{base64_str}?theme=default"

display(Image(url=url))

### Test the Classifier

Let's test with different types of content:

In [4]:
# Test 1: Positive content (should be approved)
print("=" * 60)
print("TEST 1: Positive Content")
print("=" * 60)

result1 = content_app.invoke({
    "text": "This is a great and amazing product! Excellent quality and very helpful.",
    "score": 0.0,
    "category": "",
    "message": ""
})

print(f"\n📝 Result: {result1['category'].upper()}")
print(f"💬 Message: {result1['message']}\n")

TEST 1: Positive Content
🔍 Analyzing content: 'This is a great and amazing product! Excellent qua...'
📊 Content score: 1.00
✅ Routing to APPROVE
✅ Content APPROVED

📝 Result: APPROVED
💬 Message: Content approved for publication



In [5]:
# Test 2: Negative content (should be rejected)
print("=" * 60)
print("TEST 2: Negative Content")
print("=" * 60)

result2 = content_app.invoke({
    "text": "This is spam! Terrible scam, fake product. I hate it.",
    "score": 0.0,
    "category": "",
    "message": ""
})

print(f"\n📝 Result: {result2['category'].upper()}")
print(f"💬 Message: {result2['message']}\n")

TEST 2: Negative Content
🔍 Analyzing content: 'This is spam! Terrible scam, fake product. I hate ...'
📊 Content score: 0.00
❌ Routing to REJECT
❌ Content REJECTED

📝 Result: REJECTED
💬 Message: Content does not meet quality standards



In [6]:
# Test 3: Neutral content (should be reviewed)
print("=" * 60)
print("TEST 3: Neutral Content")
print("=" * 60)

result3 = content_app.invoke({
    "text": "This product is okay. Some good features, but also some terrible aspects.",
    "score": 0.0,
    "category": "",
    "message": ""
})

print(f"\n📝 Result: {result3['category'].upper()}")
print(f"💬 Message: {result3['message']}\n")

TEST 3: Neutral Content
🔍 Analyzing content: 'This product is okay. Some good features, but also...'
📊 Content score: 0.50
⚠️  Routing to REVIEW
⚠️  Content flagged for REVIEW

📝 Result: REVIEW
💬 Message: Content requires manual review



## 4. Example 2: Grade-Based Feedback System

Let's build a more complex example with multiple decision points.

### Flow

```
START → calculate_grade → route_by_grade?
                            ├─ A: excellent_feedback → END
                            ├─ B: good_feedback → END
                            ├─ C: needs_improvement → END
                            └─ D/F: remedial_action → END
```

In [9]:
class GradeState(TypedDict):
    student_name: str
    score: int        # 0-100
    letter_grade: str # A, B, C, D, F
    feedback: str
    action_required: str

def calculate_grade(state: GradeState) -> dict:
    """Convert numerical score to letter grade."""
    score = state["score"]
    print(f"📊 Calculating grade for {state['student_name']}: {score}/100")
    
    if score >= 90:
        letter = "A"
    elif score >= 80:
        letter = "B"
    elif score >= 70:
        letter = "C"
    elif score >= 60:
        letter = "D"
    else:
        letter = "F"
    
    print(f"📝 Letter grade: {letter}")
    return {"letter_grade": letter}

def route_by_grade(state: GradeState) -> Literal["excellent", "good", "needs_work", "remedial"]:
    """Route to different feedback based on grade."""
    grade = state["letter_grade"]
    
    routes = {
        "A": "excellent",
        "B": "good",
        "C": "needs_work",
        "D": "remedial",
        "F": "remedial"
    }
    
    route = routes[grade]
    print(f"🔀 Routing to: {route.upper()}")
    return route

def excellent_feedback(state: GradeState) -> dict:
    return {
        "feedback": f"Outstanding work, {state['student_name']}! Keep up the excellent performance!",
        "action_required": "None - consider for honors program"
    }

def good_feedback(state: GradeState) -> dict:
    return {
        "feedback": f"Good job, {state['student_name']}! You're on the right track.",
        "action_required": "Continue current approach"
    }

def needs_work_feedback(state: GradeState) -> dict:
    return {
        "feedback": f"{state['student_name']}, you're passing but could improve with more practice.",
        "action_required": "Recommend extra practice sessions"
    }

def remedial_feedback(state: GradeState) -> dict:
    return {
        "feedback": f"{state['student_name']}, we need to address some fundamental gaps.",
        "action_required": "Schedule tutoring and re-assessment"
    }

In [10]:
# Build grade feedback graph
grade_graph = StateGraph(GradeState)

# Add nodes
grade_graph.add_node("calculate", calculate_grade)
grade_graph.add_node("excellent", excellent_feedback)
grade_graph.add_node("good", good_feedback)
grade_graph.add_node("needs_work", needs_work_feedback)
grade_graph.add_node("remedial", remedial_feedback)

# Add edges
grade_graph.add_edge(START, "calculate")
grade_graph.add_conditional_edges("calculate", route_by_grade)

# All terminal nodes go to END
grade_graph.add_edge("excellent", END)
grade_graph.add_edge("good", END)
grade_graph.add_edge("needs_work", END)
grade_graph.add_edge("remedial", END)

grade_app = grade_graph.compile()

print("✅ Grade feedback system ready!")

✅ Grade feedback system ready!


In [11]:
# Test with different scores
test_students = [
    {"student_name": "Alice", "score": 95},
    {"student_name": "Bob", "score": 82},
    {"student_name": "Charlie", "score": 71},
    {"student_name": "Diana", "score": 55}
]

for student in test_students:
    print("\n" + "=" * 60)
    result = grade_app.invoke({
        "student_name": student["student_name"],
        "score": student["score"],
        "letter_grade": "",
        "feedback": "",
        "action_required": ""
    })
    
    print(f"\n🎓 Grade: {result['letter_grade']}")
    print(f"💬 {result['feedback']}")
    print(f"📋 Action: {result['action_required']}")


📊 Calculating grade for Alice: 95/100
📝 Letter grade: A
🔀 Routing to: EXCELLENT

🎓 Grade: A
💬 Outstanding work, Alice! Keep up the excellent performance!
📋 Action: None - consider for honors program

📊 Calculating grade for Bob: 82/100
📝 Letter grade: B
🔀 Routing to: GOOD

🎓 Grade: B
💬 Good job, Bob! You're on the right track.
📋 Action: Continue current approach

📊 Calculating grade for Charlie: 71/100
📝 Letter grade: C
🔀 Routing to: NEEDS_WORK

🎓 Grade: C
💬 Charlie, you're passing but could improve with more practice.
📋 Action: Recommend extra practice sessions

📊 Calculating grade for Diana: 55/100
📝 Letter grade: F
🔀 Routing to: REMEDIAL

🎓 Grade: F
💬 Diana, we need to address some fundamental gaps.
📋 Action: Schedule tutoring and re-assessment


## 5. Example 3: API Response Handler

A practical example: handling different API response scenarios.

### Flow

```
START → call_api → check_status?
                     ├─ success → process_data → END
                     ├─ retryable_error → retry → call_api (loop back!)
                     └─ fatal_error → handle_error → END
```

Note: This example introduces **looping** - a node can route back to a previous node!

In [7]:
import random

class APIState(TypedDict):
    url: str
    status_code: int
    retry_count: int
    max_retries: int
    data: str
    result: str

def call_api(state: APIState) -> dict:
    """Simulate an API call that might succeed or fail."""
    print(f"🌐 Calling API: {state['url']} (Attempt {state['retry_count'] + 1})")
    
    # Simulate random responses
    outcomes = [200, 200, 500, 503]  # Weighted towards success
    status = random.choice(outcomes)
    
    print(f"📡 Response: {status}")
    
    if status == 200:
        return {"status_code": status, "data": "Success data payload"}
    else:
        return {"status_code": status, "data": ""}

def check_status(state: APIState) -> Literal["success", "retry", "error"]:
    """Route based on API response status."""
    status = state["status_code"]
    retry_count = state["retry_count"]
    max_retries = state["max_retries"]
    
    if status == 200:
        print("✅ Success! Routing to process_data")
        return "success"
    elif status in [500, 503] and retry_count < max_retries:
        print(f"⚠️  Retryable error. Routing to retry ({retry_count + 1}/{max_retries})")
        return "retry"
    else:
        print("❌ Fatal error or max retries exceeded. Routing to error handler")
        return "error"

def process_data(state: APIState) -> dict:
    """Process successful API response."""
    print("✅ Processing data...")
    return {"result": f"Processed: {state['data']}"}

def retry_request(state: APIState) -> dict:
    """Prepare for retry."""
    new_retry_count = state["retry_count"] + 1
    print(f"🔄 Preparing retry {new_retry_count}...")
    return {"retry_count": new_retry_count}

def handle_error(state: APIState) -> dict:
    """Handle fatal errors."""
    print("❌ Handling fatal error...")
    return {"result": f"Error {state['status_code']}: Request failed after {state['retry_count']} retries"}

In [8]:
# Build API handler graph
api_graph = StateGraph(APIState)

# Add nodes
api_graph.add_node("call", call_api)
api_graph.add_node("success", process_data)
api_graph.add_node("retry", retry_request)
api_graph.add_node("error", handle_error)

# Add edges
api_graph.add_edge(START, "call")
api_graph.add_conditional_edges("call", check_status)

# Success and error go to END
api_graph.add_edge("success", END)
api_graph.add_edge("error", END)

# Retry loops back to call! This creates a cycle.
api_graph.add_edge("retry", "call")

api_app = api_graph.compile()

print("✅ API handler ready!")

✅ API handler ready!


In [9]:
# Test the API handler
print("=" * 60)
print("Testing API Handler with Retries")
print("=" * 60)

result = api_app.invoke({
    "url": "https://api.example.com/data",
    "status_code": 0,
    "retry_count": 0,
    "max_retries": 3,
    "data": "",
    "result": ""
})

print("\n" + "=" * 60)
print(f"📊 Final Result: {result['result']}")
print(f"🔢 Total Attempts: {result['retry_count'] + 1}")
print("=" * 60)

Testing API Handler with Retries
🌐 Calling API: https://api.example.com/data (Attempt 1)
📡 Response: 200
✅ Success! Routing to process_data
✅ Processing data...

📊 Final Result: Processed: Success data payload
🔢 Total Attempts: 1


## 6. Hands-on Exercise: Email Triage System

Build a system that classifies and routes emails to appropriate handlers.

### Requirements

Your system should:
1. Analyze email subject and content
2. Classify into: URGENT, NORMAL, SPAM, AUTOMATED
3. Route to different handlers based on classification

### Classification Rules

- **URGENT**: Contains "urgent", "asap", "emergency"
- **SPAM**: Contains "win", "prize", "click here", "congratulations"
- **AUTOMATED**: From "noreply" or "automated"
- **NORMAL**: Everything else

### Routing

- URGENT → escalate_to_priority
- SPAM → delete_email
- AUTOMATED → auto_respond
- NORMAL → standard_processing

In [15]:
class EmailState(TypedDict):
    sender: str
    subject: str
    content: str
    classification: str
    action_taken: str

def classify_email(state: EmailState) -> dict:
    """Classify email based on content."""
    # TODO: Implement classification logic
    # Hint: Check subject and content for keywords
    pass

def route_email(state: EmailState) -> Literal["urgent", "spam", "automated", "normal"]:
    """Route email based on classification."""
    # TODO: Return the appropriate route based on classification
    pass

def escalate_to_priority(state: EmailState) -> dict:
    """Handle urgent emails."""
    # TODO: Implement urgent handler
    pass

def delete_email(state: EmailState) -> dict:
    """Handle spam."""
    # TODO: Implement spam handler
    pass

def auto_respond(state: EmailState) -> dict:
    """Handle automated emails."""
    # TODO: Implement automated handler
    pass

def standard_processing(state: EmailState) -> dict:
    """Handle normal emails."""
    # TODO: Implement normal handler
    pass

# TODO: Build the graph
# email_graph = StateGraph(EmailState)
# ... add nodes and edges ...
# email_app = email_graph.compile()

### Test Your System

In [16]:
test_emails = [
    {
        "sender": "boss@company.com",
        "subject": "URGENT: Server down!",
        "content": "We need this fixed ASAP!"
    },
    {
        "sender": "spam@lottery.com",
        "subject": "You won a prize!",
        "content": "Click here to claim your prize. Congratulations!"
    },
    {
        "sender": "noreply@service.com",
        "subject": "Account notification",
        "content": "This is an automated message."
    },
    {
        "sender": "colleague@company.com",
        "subject": "Meeting tomorrow",
        "content": "Let's discuss the project."
    }
]

# Uncomment to test:
# for email in test_emails:
#     result = email_app.invoke({
#         **email,
#         "classification": "",
#         "action_taken": ""
#     })
#     print(f"\nEmail: {result['subject']}")
#     print(f"Classification: {result['classification']}")
#     print(f"Action: {result['action_taken']}")
#     print("-" * 60)

<details>
<summary><b>Click here for the solution</b></summary>

```python
def classify_email(state: EmailState) -> dict:
    text = (state["subject"] + " " + state["content"] + " " + state["sender"]).lower()
    
    if any(word in text for word in ["urgent", "asap", "emergency"]):
        classification = "URGENT"
    elif any(word in text for word in ["win", "prize", "click here", "congratulations"]):
        classification = "SPAM"
    elif "noreply" in text or "automated" in text:
        classification = "AUTOMATED"
    else:
        classification = "NORMAL"
    
    return {"classification": classification}

def route_email(state: EmailState) -> Literal["urgent", "spam", "automated", "normal"]:
    routes = {
        "URGENT": "urgent",
        "SPAM": "spam",
        "AUTOMATED": "automated",
        "NORMAL": "normal"
    }
    return routes[state["classification"]]

def escalate_to_priority(state: EmailState) -> dict:
    return {"action_taken": "Escalated to priority queue for immediate attention"}

def delete_email(state: EmailState) -> dict:
    return {"action_taken": "Moved to spam folder"}

def auto_respond(state: EmailState) -> dict:
    return {"action_taken": "Auto-response sent, email archived"}

def standard_processing(state: EmailState) -> dict:
    return {"action_taken": "Added to inbox for standard processing"}

email_graph = StateGraph(EmailState)
email_graph.add_node("classify", classify_email)
email_graph.add_node("urgent", escalate_to_priority)
email_graph.add_node("spam", delete_email)
email_graph.add_node("automated", auto_respond)
email_graph.add_node("normal", standard_processing)

email_graph.add_edge(START, "classify")
email_graph.add_conditional_edges("classify", route_email)
email_graph.add_edge("urgent", END)
email_graph.add_edge("spam", END)
email_graph.add_edge("automated", END)
email_graph.add_edge("normal", END)

email_app = email_graph.compile()
```
</details>

## 🆕 LangGraph 1.1.9: The `Command` Class — A Concise Alternative

LangGraph 1.1.9 introduced the `Command` class as a cleaner alternative to routing functions + `add_conditional_edges` for simple cases.

**Traditional approach** (always valid, covered throughout this notebook):
```python
def route(state: State) -> Literal["approve", "reject"]:
    return "approve" if state["score"] >= 0.7 else "reject"

graph.add_conditional_edges("analyze", route)
```

**Command approach** (new, concise):
```python
from langgraph.types import Command

def analyze(state: State) -> Command:
    score = compute_score(state["text"])
    return Command(
        update={"score": score},        # state update
        goto="approve" if score >= 0.7 else "reject",  # routing
    )
# No add_conditional_edges needed — Command handles both!
```

**When to use Command vs conditional edges:**
| Situation | Use |
|---|---|
| Routing depends on expensive computation already done in the node | `Command` (avoids re-computing in routing function) |
| Routing logic is complex with many paths | `add_conditional_edges` (clearer separation) |
| State update + routing happen together | `Command` |
| Static routing that doesn't change state | `add_conditional_edges` |

> **Deep dive**: `Command` has more superpowers (HITL resumption, subgraph escape). We cover it fully in **Notebook 08**.

In [17]:
# Command example — conditional routing + state update in one return
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import Literal
from typing_extensions import TypedDict

class ContentState(TypedDict):
    text: str
    score: float
    result: str

def analyze_and_route(state: ContentState) -> Command:
    """Analyze content and route — combined in one node."""
    # Simulate scoring
    score = len(state["text"]) / 100  # simplified score
    score = min(score, 1.0)

    if score >= 0.7:
        destination = "approve"
    elif score >= 0.3:
        destination = "review"
    else:
        destination = "reject"

    print(f"📊 Score: {score:.2f} → routing to '{destination}'")
    return Command(update={"score": score}, goto=destination)

def approve(state: ContentState) -> dict:
    return {"result": f"✅ APPROVED (score: {state['score']:.2f})"}

def review(state: ContentState) -> dict:
    return {"result": f"🔍 NEEDS REVIEW (score: {state['score']:.2f})"}

def reject(state: ContentState) -> dict:
    return {"result": f"❌ REJECTED (score: {state['score']:.2f})"}

# Build graph — no add_conditional_edges needed!
graph = StateGraph(ContentState)
graph.add_node("analyze", analyze_and_route)
graph.add_node("approve", approve)
graph.add_node("review", review)
graph.add_node("reject", reject)

graph.add_edge(START, "analyze")
graph.add_edge("approve", END)
graph.add_edge("review", END)
graph.add_edge("reject", END)

app = graph.compile()

# Test it
result = app.invoke({
    "text": "This is a long and detailed piece of content that deserves careful review and consideration.",
    "score": 0.0,
    "result": ""
})
print(result["result"])

📊 Score: 0.92 → routing to 'approve'
✅ APPROVED (score: 0.92)


## 7. Key Takeaways

### New Concepts Mastered

1. **Conditional Edges**: Dynamic routing based on state
2. **Routing Functions**: Functions that return next node name or END
3. **Branching Workflows**: Multiple execution paths from one node
4. **Cyclic Graphs**: Edges can create loops for retry logic
5. **Literal Types**: Type hints for routing function returns

### API Reference

```python
# Add conditional edge
graph.add_conditional_edges(
    source="source_node",
    path=routing_function,  # Returns node name or END
)

# Routing function signature
def routing_function(state: State) -> Literal["node1", "node2", ...]:
    # Decision logic here
    return "node1"  # or "node2", etc.
```

### Best Practices

✅ **Use Literal types** for routing function return values  
✅ **Handle all cases** in your routing logic  
✅ **Add logging** to understand routing decisions  
✅ **Limit retry loops** to prevent infinite cycles  
✅ **Make routing deterministic** - same state should always route the same way  

### Common Pitfalls

❌ Routing function returns invalid node name  
❌ Forgetting to add edges for all possible routes  
❌ Creating infinite loops without exit conditions  
❌ Non-deterministic routing (using random without seed)  
❌ Complex routing logic that's hard to test  

### What's Next?

In **Notebook 03: Tool Integration and OpenAI Function Calling**, you'll learn:

- Integrating OpenAI GPT models into your graphs
- Tool/function calling with LLMs
- Building AI agents that can use tools
- Message state management
- Practical agent patterns

This is where LangGraph truly shines - building intelligent agents!

## 8. Further Reading

- [Conditional Edges Documentation](https://langchain-ai.github.io/langgraph/how-tos/branching/)
- [Graph Visualization](https://langchain-ai.github.io/langgraph/how-tos/visualization/)
- [Cycles and Recursion](https://langchain-ai.github.io/langgraph/concepts/low_level/#cycles)

### Challenge Exercises

1. **Multi-language Router**: Route text to different translators based on detected language
2. **Dynamic Retry System**: Implement exponential backoff with conditional routing
3. **Content Moderator**: Multi-level moderation with escalation paths

---

**Ready for AI agents?** Continue to [Notebook 03: Tool Integration and OpenAI Function Calling](03_tool_integration_openai.ipynb)!

### Parallel Edges - FAN IN FAN OUT

In [18]:
import operator
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

# ------------------------------------------------------------
# LLM
# ------------------------------------------------------------

llm = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0.7,
)

# ------------------------------------------------------------
# STATE
# ------------------------------------------------------------

class State(TypedDict):
    topic: str
    sections: Annotated[list[str], operator.add]
    final_post: str

# ------------------------------------------------------------
# NODES
# ------------------------------------------------------------

def prepare_topic(state: State):
    print(f"\n📋 Preparing blog about: {state['topic']}")
    return {}

# ---------------- PARALLEL NODE 1 ----------------

def write_intro(state: State):

    prompt = f"""
    Write an engaging blog introduction about:
    {state['topic']}

    Keep it concise and professional.
    """

    response = llm.invoke(prompt)

    return {
        "sections": [f"# Introduction\n\n{response.content}"]
    }

# ---------------- PARALLEL NODE 2 ----------------

def write_body(state: State):

    prompt = f"""
    Write the main body of a blog about:
    {state['topic']}

    Include:
    - Top use cases
    - Industry applications
    - Real-world impact

    Use markdown formatting.
    """

    response = llm.invoke(prompt)

    return {
        "sections": [f"# Main Content\n\n{response.content}"]
    }

# ---------------- PARALLEL NODE 3 ----------------

def write_conclusion(state: State):

    prompt = f"""
    Write a strong conclusion for a blog about:
    {state['topic']}

    Mention future implications and trends.
    """

    response = llm.invoke(prompt)

    return {
        "sections": [f"# Conclusion\n\n{response.content}"]
    }

# ------------------------------------------------------------
# FAN-IN
# ------------------------------------------------------------

def assemble_post(state: State):

    print("\n🔗 Combining all sections...")

    full_post = "\n\n".join(state["sections"])

    return {
        "final_post": full_post
    }

# ------------------------------------------------------------
# GRAPH
# ------------------------------------------------------------

builder = StateGraph(State)

builder.add_node("prepare_topic", prepare_topic)

builder.add_node("write_intro", write_intro)
builder.add_node("write_body", write_body)
builder.add_node("write_conclusion", write_conclusion)

builder.add_node("assemble_post", assemble_post)

builder.add_edge(START, "prepare_topic")

# FAN-OUT
builder.add_edge("prepare_topic", "write_intro")
builder.add_edge("prepare_topic", "write_body")
builder.add_edge("prepare_topic", "write_conclusion")

# FAN-IN
builder.add_edge("write_intro", "assemble_post")
builder.add_edge("write_body", "assemble_post")
builder.add_edge("write_conclusion", "assemble_post")

builder.add_edge("assemble_post", END)

# ------------------------------------------------------------
# COMPILE
# ------------------------------------------------------------

graph = builder.compile()

# ------------------------------------------------------------
# RUN
# ------------------------------------------------------------

result = graph.invoke({
    "topic": "How Agentic AI is Transforming Software Engineering",
    "sections": [],
    "final_post": "",
})

print("\n📝 FINAL BLOG:\n")
print(result["final_post"])


📋 Preparing blog about: How Agentic AI is Transforming Software Engineering

🔗 Combining all sections...

📝 FINAL BLOG:

# Main Content

How Agentic AI is Transforming Software Engineering

Agentic AI refers to AI systems that aren’t just passive tools but active agents capable of making decisions, taking actions, and coordinating tasks to achieve defined goals. In software engineering, this shift from “AI as a helper” to “AI as an agent within the workflow” is accelerating development velocity, improving reliability, and enabling engineers to focus on higher-value work. Below is a practical look at what’s changing, where it’s most impactful, and what teams should know as they adopt agentic AI.

## What makes agentic AI different for software engineering

- Autonomy with guardrails: Agents operate end-to-end on well-defined goals but require human-in-the-loop safety checks for high-stakes actions (shipping code, changing production configurations, etc.).
- Goal-driven collaboration: M